In [1]:
import fastf1
import pandas as pd
import numpy as np

# Enable FastF1 cache
fastf1.Cache.enable_cache('D:\\Programming\\Python\\AI\\F1 Fantasy\\cache')

In [2]:
# Factor to increase weight of practice pace compared to qpace and historical rpace
# Smaller value means greater weight
practiceWeightFactor = 0.7

practice_rpace = pd.read_csv("practice_rpace.csv", index_col=0).squeeze('columns')
#practice_rpace['sigma'] *= practiceWeightFactor

qualiWeightFactor = 0.4
current_qpace = pd.read_csv("current_qpace.csv", index_col=0).squeeze('columns')
#current_qpace *= qualiWeightFactor

historical_rpace = pd.read_csv("historical_rpace.csv", index_col=0).squeeze('columns')
current_qpace

,mu,sigma
ALB,0.024706,0.002162
ALO,0.021504,0.002183
ANT,0.003405,0.002164
BEA,0.022800,0.002156
BOR,0.016213,0.002164
BOT,0.033051,0.002171
COL,0.042841,0.002160
GAS,0.013662,0.002157
HAD,0.003822,0.002188
HAM,0.005290,0.002168


In [3]:
def blend_sources(sources, tau):
    # sources: dict name -> (mu, sigma)   sigma = sampling SE, fractional units
    # tau: drift sd added
    names, mu, var = [], [], []
    for name, (m, s) in sources.items():
        if not np.isfinite(m) and np.isfinite(s):
            continue
        names.append(name)
        mu.append(m)
        var.append(s**2 + tau.get(name, 0.0)**2)

    if not names:
        return np.nan, np.inf, {}

    mu, var = np.array(mu), np.array(var)
    w = 1.0 / var
    blended_mu = float(np.sum(w * mu) / w.sum())
    blended_sigma = float(np.sqrt(1.0 / w.sum()))
    return blended_mu, blended_sigma, dict(zip(names, w / w.sum()))

In [4]:
# NEW VERSION
TAU_R = {'practice': 0.0030, 'race_form': 0.0040, 'quali_form': 0.0030}

def blend_rpace_tau(
    driver, practice_rpace, current_qpace, historical_rpace, tau=TAU_R
):
    sources = {}
    if driver in practice_rpace.index:
        sources['practice'] = (practice_rpace.loc[driver]['mu'], practice_rpace.loc[driver]['sigma'])
    else:
        sources['practice'] = (np.nan, np.inf)

    sources['race_form'] = (historical_rpace.loc[driver]['mu'], historical_rpace.loc[driver]['sigma'])
    sources['quali_form'] = (current_qpace.loc[driver]['mu'], current_qpace.loc[driver]['sigma'])

    mu, sigma, shares = blend_sources(sources, tau)
    return mu, sigma, shares, sources

In [5]:
# NEW VERSION
predicted_rpace, predicted_rpace_sigma, rpace_shares, round_srcs = {}, {}, {}, {}
for d in current_qpace.index:
    mu, sig, shares, srcs = blend_rpace_tau(d, practice_rpace, current_qpace, historical_rpace)
    predicted_rpace[d], predicted_rpace_sigma[d] = mu, sig
    rpace_shares[d], round_srcs[d] = shares, srcs

predicted_rpace = pd.Series(predicted_rpace).sort_values()
predicted_rpace_sigma = pd.Series(predicted_rpace_sigma)
shares_df = pd.DataFrame(rpace_shares).T
print(shares_df.mean())          # average weight each source carries
print(shares_df.round(2).loc[['ANT', 'VER', 'STR']])
predicted_rpace = pd.concat([predicted_rpace, predicted_rpace_sigma], axis=1, keys=['mu', 'sigma'])
predicted_rpace

practice      0.366677
race_form     0.287295
quali_form    0.346028
dtype: float64
     practice  race_form  quali_form
ANT      0.37       0.29        0.34
VER      0.36       0.29        0.35
STR      0.36       0.29        0.35


,mu,sigma
RUS,0.002427,0.002187
VER,0.003051,0.002181
LEC,0.004722,0.002155
PIA,0.005139,0.002187
NOR,0.005557,0.002178
HAM,0.007318,0.002185
ANT,0.007395,0.002169
HAD,0.010057,0.002170
GAS,0.014218,0.002191
HUL,0.019216,0.002179


In [6]:
predicted_rpace.to_csv("predicted_rpace.csv")

In [52]:
# OLD VERSION
def blend_predicted_rpace(driver, practice_rpace, historical_rpace, current_qpace):
    sources = []

    # Source 1: practice
    if driver in practice_rpace.index:
        sources.append((practice_rpace.loc[driver]['mu'], practice_rpace.loc[driver]['sigma']))
    else:
        sources.append((0.0, np.inf))
    
    # Source 2: recency-weighted historical race form
    sources.append((historical_rpace.loc[driver]['mu'], historical_rpace.loc[driver]['sigma']))

    # Source 3: predicted qpace (using practice and historical data)
    sources.append((current_qpace.loc[driver]['mu'], current_qpace.loc[driver]['sigma']))

    mu = np.array([s[0] for s in sources])
    sigma = np.array([s[1] for s in sources])
    w = 1.0 / sigma**2

    if not np.isfinite(w).any():
        raise ValueError(f"No Rpace source at all for {driver}")

    blended_mu = np.sum(w * mu) / np.sum(w)
    blended_sigma = np.sqrt(1.0 / np.sum(w))

    breakdown = dict(zip(['practice', 'race_form', 'quali'], zip(mu, sigma)))
    return blended_mu, blended_sigma, breakdown

In [59]:
# OLD VERSION
current_round = 11

predicted_rpace = {}
predicted_rpace_sigma = {}
breakdowns = {}

for driver in historical_rpace.index:
    mu, sigma, breakdown = blend_predicted_rpace(driver, practice_rpace, historical_rpace, current_qpace)
    predicted_rpace[driver] = mu
    predicted_rpace_sigma[driver] = sigma
    breakdowns[driver] = breakdown

predicted_rpace = pd.Series(predicted_rpace).sort_values()
predicted_rpace_sigma = pd.Series(predicted_rpace_sigma)

predicted_rpace = pd.concat([predicted_rpace, predicted_rpace_sigma], axis=1, keys=['mu', 'sigma'])
predicted_rpace

,mu,sigma
ANT,0.001335,0.000511
LEC,0.003477,0.000496
RUS,0.003773,0.000516
HAM,0.003834,0.000491
NOR,0.005014,0.000533
PIA,0.005922,0.000572
VER,0.007059,0.000492
HAD,0.010539,0.000545
HUL,0.016426,0.000528
LAW,0.016705,0.000502


In [46]:
rows = []
for d, b in breakdowns.items():
    row = {'driver': d, 'blended_mu': predicted_rpace[d], 'blended_sigma': predicted_rpace_sigma[d]}
    for src_name, (mu, sig) in b.items():
        row[f'{src_name}_mu'] = mu
        row[f'{src_name}_sigma'] = sig
    rows.append(row)

diag = pd.DataFrame(rows).set_index('driver').sort_values('blended_mu')
print(diag.round(4))

        blended_mu  blended_sigma  practice_mu  practice_sigma  race_form_mu  \
driver                                                                         
ANT         0.0013         0.0005       0.0174          0.0043        0.0004   
LEC         0.0035         0.0005       0.0085          0.0041        0.0048   
RUS         0.0038         0.0005       0.0043          0.0041        0.0032   
HAM         0.0038         0.0005       0.0000          0.0037        0.0053   
NOR         0.0050         0.0005       0.0139          0.0038        0.0065   
PIA         0.0059         0.0006       0.0108          0.0041        0.0064   
VER         0.0071         0.0005       0.0153          0.0036        0.0091   
HAD         0.0105         0.0005       0.0123          0.0039        0.0153   
HUL         0.0164         0.0005       0.0161          0.0038        0.0198   
LAW         0.0167         0.0005       0.0127          0.0036        0.0214   
LIN         0.0169         0.0005       